# Agent Protocols: MCP & A2A

Companion notebook for the [Agent Protocols wiki page](https://ml-viz.vercel.app/wiki/agent-protocols-mcp-a2a).

To make the two protocols concrete without any network or SDK, we implement them **in-process** over
a mock JSON-RPC transport:

1. a minimal **MCP** server/client (`initialize`, `tools/list`, `tools/call`), and
2. a minimal **A2A** exchange (Agent Card discovery + a `Task` with a lifecycle),

then compose them: an orchestrator uses A2A to delegate to a specialist agent that itself uses MCP
tools. No API keys, no sockets — just dictionaries shaped like the real wire format.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import itertools

_ids = itertools.count(1)
def rpc(method, **params):
    """Build a JSON-RPC 2.0 request envelope (the shared wire format of both protocols)."""
    return {"jsonrpc": "2.0", "id": next(_ids), "method": method, "params": params}

## 1 — A minimal MCP server

An MCP server exposes **tools** (name + JSON-Schema + handler) and answers two methods: `tools/list`
for discovery and `tools/call` to invoke one. This is the vertical edge — agent ↔ tools/data.

In [ ]:
class MCPServer:
    def __init__(self, name):
        self.name = name
        self._tools = {}      # name -> (schema, handler)

    def tool(self, name, description, schema):
        def deco(fn):
            self._tools[name] = ({"name": name, "description": description,
                                  "inputSchema": schema}, fn)
            return fn
        return deco

    def handle(self, req):
        m = req["method"]
        if m == "initialize":
            return {"result": {"serverInfo": {"name": self.name}, "capabilities": {"tools": {}}}}
        if m == "tools/list":
            return {"result": {"tools": [meta for meta, _ in self._tools.values()]}}
        if m == "tools/call":
            name = req["params"]["name"]
            args = req["params"].get("arguments", {})
            _, fn = self._tools[name]
            return {"result": {"content": [{"type": "text", "text": str(fn(**args))}]}}
        return {"error": {"code": -32601, "message": f"method not found: {m}"}}

# A 'billing tools' server exposing one tool
billing_tools = MCPServer("billing-tools")

@billing_tools.tool("refund", "Issue a refund for an order id.",
                    {"type": "object", "properties": {"order_id": {"type": "string"}},
                     "required": ["order_id"]})
def _refund(order_id):
    return {"order_id": order_id, "status": "refunded", "amount": 42.00}

## 2 — An MCP client talking to the server

The client does the `initialize` handshake, discovers tools, then calls one — exactly the sequence
from the wiki page, just with the transport being a direct method call instead of stdio/HTTP.

In [ ]:
class MCPClient:
    def __init__(self, server):
        self.server = server
        self.server.handle(rpc("initialize"))      # handshake
    def list_tools(self):
        return self.server.handle(rpc("tools/list"))["result"]["tools"]
    def call(self, name, **arguments):
        resp = self.server.handle(rpc("tools/call", name=name, arguments=arguments))
        return resp["result"]["content"][0]["text"]

client = MCPClient(billing_tools)
print("discovered tools:", [t["name"] for t in client.list_tools()])
print("refund result:   ", client.call("refund", order_id="4471"))

## 3 — A minimal A2A agent (card + task lifecycle)

An A2A agent publishes an **Agent Card** (capability discovery) and accepts **Tasks** that move
through a lifecycle (`submitted → working → completed`). This is the horizontal edge — agent ↔ agent.
Our billing agent fulfils tasks by calling its *own* MCP tool server — so the two protocols compose.

In [ ]:
class A2AAgent:
    def __init__(self, name, skills, fulfil):
        self.name, self.skills, self._fulfil = name, skills, fulfil
    @property
    def agent_card(self):                       # served at /.well-known/agent.json
        return {"name": self.name, "skills": self.skills,
                "endpoint": f"https://{self.name}.example.com/a2a"}
    def handle(self, req):                      # method: message/send
        text = req["params"]["message"]["parts"][0]["text"]
        states = ["submitted", "working"]
        artifact = self._fulfil(text)           # do the work (may call MCP tools)
        states.append("completed")
        return {"result": {"task": {"states": states, "status": "completed",
                                    "artifact": artifact}}}

def billing_fulfil(text):
    order_id = text.split("#")[-1].strip()
    return client.call("refund", order_id=order_id)   # A2A task -> MCP tool call

billing_agent = A2AAgent("billing", skills=["refunds", "invoices"], fulfil=billing_fulfil)
print("agent card:", billing_agent.agent_card)

## 4 — Orchestrator delegates over A2A → which uses MCP

The orchestrator discovers the billing agent's card, checks it has the needed skill, then sends a
task. The billing agent fulfils it by calling its MCP `refund` tool. `N×M` integrations collapse to
`N+M`: the orchestrator speaks A2A once, the tool is wrapped in MCP once.

In [ ]:
def orchestrate(remote_agent, request_text, needed_skill):
    card = remote_agent.agent_card                       # 1. discovery
    assert needed_skill in card["skills"], "agent lacks the skill"
    req = rpc("message/send",
              message={"role": "user", "parts": [{"kind": "text", "text": request_text}]})
    task = remote_agent.handle(req)["result"]["task"]     # 2. delegate
    return task

task = orchestrate(billing_agent, "please refund order #4471", needed_skill="refunds")
print("task lifecycle:", " → ".join(task["states"]))
print("final status:  ", task["status"])
print("artifact:      ", task["artifact"])

## ✏️ Your turn

**Exercise.** Real systems must not blindly invoke whatever a peer claims. Implement
`safe_orchestrate(remote_agent, request_text, needed_skill, allowed_agents)` that delegates **only**
if (a) the agent's name is in `allowed_agents` (least-privilege / trust check) **and** (b) its Agent
Card advertises `needed_skill`. Otherwise return `{"status": "rejected", "reason": ...}` without
calling the agent — the security posture the wiki page argues for.

In [ ]:
def safe_orchestrate(remote_agent, request_text, needed_skill, allowed_agents):
    card = remote_agent.agent_card
    # TODO(you): reject (without calling handle) if the agent isn't allowed or lacks the skill;
    #            otherwise delegate and return the completed task dict
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
allow = {"billing"}
ok = safe_orchestrate(billing_agent, "refund order #99", "refunds", allow)
assert ok["status"] == "completed"

untrusted = A2AAgent("rogue", skills=["refunds"], fulfil=billing_fulfil)
blocked = safe_orchestrate(untrusted, "refund order #99", "refunds", allow)
assert blocked["status"] == "rejected", "untrusted agent must be blocked"

no_skill = safe_orchestrate(billing_agent, "forecast revenue", "forecasting", allow)
assert no_skill["status"] == "rejected", "missing skill must be rejected"
print("✓ delegates only to a trusted agent that advertises the needed skill")

<details>
<summary>Solution</summary>

```python
def safe_orchestrate(remote_agent, request_text, needed_skill, allowed_agents):
    card = remote_agent.agent_card
    if card["name"] not in allowed_agents:
        return {"status": "rejected", "reason": "agent not in allowlist"}
    if needed_skill not in card["skills"]:
        return {"status": "rejected", "reason": "skill not advertised"}
    req = rpc("message/send",
              message={"role": "user", "parts": [{"kind": "text", "text": request_text}]})
    return remote_agent.handle(req)["result"]["task"]
```

Discovery (the Agent Card) tells you what a peer *claims* it can do; the allowlist encodes who you
actually *trust*. A standardized protocol makes integration easy for everyone — including bad
actors — so authentication and least-privilege are not optional add-ons.

</details>